In [ ]:
import json
from tracemalloc import take_snapshot

def flip_label(t: str) -> str:
    if t == '→': return '←'
    if t == '←': return '→'
    return take_snapshot

def order_edges_by_walk(walk, edges):
    dir_map = {}
    for u, t, v in edges:
        dir_map[(u, v)] = t
        dir_map[(v, u)] = flip_label(t)

    ordered = []
    for i in range(len(walk) - 1):
        u, v = walk[i], walk[i+1]
        t = dir_map.get((u, v))
        if t is None:
            found = False
            for a, tt, b in edges:
                if a == u and b == v:
                    t = tt
                    found = True
                    break
                if a == v and b == u:
                    t = flip_label(tt)
                    found = True
                    break
            if not found:
                raise ValueError(f"Edge not found for consecutive pair {u} -> {v}")
        ordered.append([u, t, v])
    return ordered

def reorder_edges_in_jsonl(in_file, out_file, replace=False):
    n = 0
    with open(in_file, "r", encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
        for line in fin:
            n += 1
            rec = json.loads(line)
            walk = rec.get("walk")
            edges = rec.get("edges")
            if not walk or not edges:
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            try:
                edges_ordered = order_edges_by_walk(walk, edges)
            except Exception as e:
                rec["edges_order_error"] = str(e)
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            if replace:
                rec["edges"] = edges_ordered
            else:
                rec["edges_ordered"] = edges_ordered

            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Processed {n} records -> {out_file}")

In [5]:
reorder_edges_in_jsonl("path2/paths2_dist_5.jsonl", "path2/paths2_dist_5_ordered.jsonl", replace=True)

Processed 2000 records -> path2/paths2_dist_5_ordered.jsonl
